In [1]:
# Cell 1: Install Additional Packages for GUI
!pip install gradio moviepy pydub


In [2]:
# Cell 2: Import Libraries
import gradio as gr
import torch
import librosa
import numpy as np
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from moviepy.editor import VideoFileClip
import tempfile
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print("✓ Libraries imported successfully!")


✓ Libraries imported successfully!


In [3]:
# Cell 3: Load Trained Model
print("Loading your fine-tuned model...")

model_path = r"C:\Users\koust\AnaKonda\ASK\VIDEO_TO_TEXT\whisper-finetuned-final"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

processor = WhisperProcessor.from_pretrained(model_path)
model = WhisperForConditionalGeneration.from_pretrained(model_path)
model = model.to(device)
model.eval()

Loading your fine-tuned model...


WhisperForConditionalGeneration(
  (model): WhisperModel(
    (encoder): WhisperEncoder(
      (conv1): Conv1d(80, 768, kernel_size=(3,), stride=(1,), padding=(1,))
      (conv2): Conv1d(768, 768, kernel_size=(3,), stride=(2,), padding=(1,))
      (embed_positions): Embedding(1500, 768)
      (layers): ModuleList(
        (0-11): 12 x WhisperEncoderLayer(
          (self_attn): WhisperAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=False)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out_features=768, bias=True)
          (f

In [4]:
# Cell 4: Video to Audio Extraction Function
def extract_audio_from_video(video_path):
    """Extract audio from video file"""
    try:
        # Create temporary audio file
        temp_audio = tempfile.NamedTemporaryFile(delete=False, suffix='.wav')
        temp_audio_path = temp_audio.name
        temp_audio.close()
        
        # Extract audio using moviepy
        video = VideoFileClip(video_path)
        video.audio.write_audiofile(temp_audio_path, fps=16000, nbytes=2, codec='pcm_s16le', verbose=False, logger=None)
        video.close()
        
        return temp_audio_path
    except Exception as e:
        raise Exception(f"Error extracting audio: {str(e)}")

In [5]:
# Cell 5: Transcription Function
def transcribe_video(video_file, progress=gr.Progress()):
    """
    Main transcription function for video files
    """
    if video_file is None:
        return "⚠️ Please upload a video file first!"
    
    try:
        progress(0.1, desc="Extracting audio from video...")
        
        # Extract audio from video
        audio_path = extract_audio_from_video(video_file)
        
        progress(0.3, desc="Loading audio...")
        
        # Load and preprocess audio
        audio_array, sampling_rate = librosa.load(audio_path, sr=16000)
        
        # Split audio into chunks if too long (30 seconds chunks)
        chunk_length = 30 * 16000  # 30 seconds
        chunks = []
        
        if len(audio_array) > chunk_length:
            progress(0.4, desc="Splitting audio into chunks...")
            num_chunks = int(np.ceil(len(audio_array) / chunk_length))
            for i in range(num_chunks):
                start = i * chunk_length
                end = min((i + 1) * chunk_length, len(audio_array))
                chunks.append(audio_array[start:end])
        else:
            chunks = [audio_array]
        
        # Transcribe each chunk
        transcriptions = []
        for i, chunk in enumerate(chunks):
            progress(0.4 + (0.5 * (i + 1) / len(chunks)), 
                    desc=f"Transcribing chunk {i+1}/{len(chunks)}...")
            
            # Process audio
            input_features = processor(
                chunk, 
                sampling_rate=16000, 
                return_tensors="pt"
            ).input_features
            
            input_features = input_features.to(device)
            
            # Generate transcription
            with torch.no_grad():
                predicted_ids = model.generate(
                    input_features,
                    max_length=225,
                    num_beams=5,
                    early_stopping=True
                )
            
            # Decode
            transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
            transcriptions.append(transcription)
        
        progress(0.95, desc="Finalizing...")
        
        # Combine all transcriptions
        full_transcription = " ".join(transcriptions)
        
        # Clean up temporary file
        if os.path.exists(audio_path):
            os.remove(audio_path)
        
        progress(1.0, desc="Complete!")
        
        # Format output
        output = f"""
📝 TRANSCRIPTION COMPLETE
{'='*60}

{full_transcription}

{'='*60}
📊 Statistics:
• Audio Duration: {len(audio_array)/16000:.2f} seconds
• Chunks Processed: {len(chunks)}
• Words Transcribed: {len(full_transcription.split())}
• Device Used: {device}
"""
        return output
        
    except Exception as e:
        return f"❌ Error: {str(e)}\n\nPlease make sure:\n• Video file is valid\n• Video contains audio\n• File format is supported (mp4, avi, mov, mkv)"


In [6]:
# Cell 6: Create Gradio Interface
print("Creating GUI interface...")

# Custom CSS for better styling
custom_css = """
.gradio-container {
    font-family: 'Arial', sans-serif;
}
.output-text {
    font-size: 16px;
    line-height: 1.6;
}
"""

# Create interface
with gr.Blocks(css=custom_css, title="Video Transcription AI", theme=gr.themes.Soft()) as demo:
    
    gr.Markdown(
        """
        # 🎬 Video-to-Text Transcription AI
        ### Upload a video file and get accurate text transcription powered by your fine-tuned Whisper model
        """
    )
    
    with gr.Row():
        with gr.Column(scale=1):
            # Input section
            gr.Markdown("### 📤 Upload Video")
            video_input = gr.Video(
                label="Select Video File",
                sources=["upload"],
            )
            
            gr.Markdown(
                """
                **Supported formats:** MP4, AVI, MOV, MKV, WebM  
                **Max duration:** No limit (longer videos take more time)
                """
            )
            
            transcribe_btn = gr.Button("🎙️ Start Transcription", variant="primary", size="lg")
            
            gr.Markdown("---")
            
            # Example section
            gr.Markdown("### 💡 Tips")
            gr.Markdown(
                """
                • Clear audio produces better results
                • Longer videos are split into 30-second chunks
                • Processing time: ~2-3 seconds per 30 seconds of audio
                • GPU acceleration enabled on your RTX 5070
                """
            )
        
        with gr.Column(scale=1):
            # Output section
            gr.Markdown("### 📄 Transcription Output")
            output_text = gr.Textbox(
                label="Transcribed Text",
                placeholder="Your transcription will appear here...",
                lines=20,
                max_lines=30,
                elem_classes="output-text"
            )
            
            # Download button
            download_btn = gr.File(label="💾 Download Transcription", visible=False)
    
    # Event handler
    def transcribe_and_save(video_file, progress=gr.Progress()):
        result = transcribe_video(video_file, progress)
        
        # Save to file
        if not result.startswith("❌") and not result.startswith("⚠️"):
            temp_file = tempfile.NamedTemporaryFile(delete=False, suffix='.txt', mode='w', encoding='utf-8')
            temp_file.write(result)
            temp_file.close()
            return result, temp_file.name
        
        return result, None
    
    transcribe_btn.click(
        fn=transcribe_and_save,
        inputs=[video_input],
        outputs=[output_text, download_btn]
    )
    
    gr.Markdown(
        """
        ---
        ### 🔧 Model Information
        **Model:** Fine-tuned Whisper  
        **Framework:** PyTorch  
        **Device:** CUDA (RTX 5070)  
        **Languages:** English
        """
    )

print("✓ GUI interface created!")


Creating GUI interface...
✓ GUI interface created!


In [ ]:
# Cell 7: Launch the Interface
print("\n" + "="*60)
print("🚀 LAUNCHING VIDEO TRANSCRIPTION GUI")
print("="*60)
print("\nStarting server...")

# Launch with specific settings
demo.launch(
    share=True,  # Create public link
    server_name="0.0.0.0",  # Allow external connections
    server_port=7860,  # Port number
    show_error=True,
    debug=True
)

# Note: A public URL will be generated that you can share
# The interface will open automatically in your browser


🚀 LAUNCHING VIDEO TRANSCRIPTION GUI

Starting server...
* Running on local URL:  http://0.0.0.0:7860
* Running on public URL: https://b374dc8ed79327ed77.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
`generation_config` default values have been modified to match model-specific defaults: {'suppress_tokens': [1, 2, 7, 8, 9, 10, 14, 25, 26, 27, 28, 29, 31, 58, 59, 60, 61, 62, 63, 90, 91, 92, 93, 359, 503, 522, 542, 873, 893, 902, 918, 922, 931, 1350, 1853, 1982, 2460, 2627, 3246, 3253, 3268, 3536, 3846, 3961, 4183, 4667, 6585, 6647, 7273, 9061, 9383, 10428, 10929, 11938, 12033, 12331, 12562, 13793, 14157, 14635, 15265, 15618, 16553, 16604, 18362, 18956, 20075, 21675, 22520, 26130, 26161